# Ocean Acidification Extreme calculation 

**** New OAX diagnostic based on Bach term

Using this notebook to compute OAX diagnostics (duration and mean magnitude)

Using papermill1 to run in batch and can be applied to the prescribed oax_var passed to routine.

Applied to the OFAM future simulation with statics compute for various GWLs

Did the calculations with megamem 


In [1]:
import xarray as xr
from dask.distributed import Client
import dask.array as da
from dask import delayed
import numpy as np
import os
import matplotlib.pyplot as plt
from glob import glob
import warnings
# importing sys
import sys

import dask
# Set configuration options
dask.config.set({
    'distributed.comm.timeouts.connect': '90s',  # Timeout for connecting to a worker
    'distributed.comm.timeouts.tcp': '90s',  # Timeout for TCP communications
})

In [2]:
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 28,Total memory: 125.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39397,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46737,Total threads: 4
Dashboard: /proxy/39097/status,Memory: 17.88 GiB
Nanny: tcp://127.0.0.1:35139,


In [3]:
def drop_stuff(ds, coords_to_drop, vars_to_drop):
    ds = ds.drop_vars(coords_to_drop, errors='ignore')
    ds = ds.drop_vars(vars_to_drop, errors='ignore')
    return ds

In [4]:
def load_data1(oax_var, coords_to_drop, vars_to_drop, pattern0, pattern1, pattern2):

    dsst1 = xr.open_mfdataset(pattern0, parallel=True, preprocess=lambda x: drop_stuff(x, coords_to_drop, vars_to_drop)).squeeze()
    dsst2 = xr.open_mfdataset(pattern1 + pattern2, parallel=True, chunks={'TIME41': 31},preprocess=lambda x: drop_stuff(x, coords_to_drop, vars_to_drop)).squeeze()
#rename coordinates!!
    dsst2 = dsst2.rename({"TIME41": "time","XT_OCEAN": "xt_ocean", "YT_OCEAN": "yt_ocean" })
    dsst1 = dsst1.rename({"TIME41": "time","XT_OCEAN": "xt_ocean", "YT_OCEAN": "yt_ocean" })

    sst = dsst2[oax_var]
    ssth= dsst1[oax_var]
    return sst,ssth

In [5]:
def calculate_heatwave_metrics(sst, threshold90, period, start, end):
    sst_period = sst.sel(time=slice(start, end))
    nshape = np.shape(sst_period)
    nyears = int(nshape[0] / 365.25 + .5)

    expanded_threshold = threshold90.sel(dayofyear=sst_period['time'].dt.dayofyear)
    expanded_threshold = expanded_threshold.drop('dayofyear')

    diff = sst_period - expanded_threshold

    min_dur = 10
    hmask = sst_period > expanded_threshold
    hmask1 = hmask.rolling(time=min_dur, center=False).sum() >= min_dur
    hmask2 = hmask1.shift(time=((min_dur-1)*-1), fill_value=False)
    hmask3 = hmask1 | hmask2

    dur_mhw = (hmask3.sum('time') / nyears)
    mag_mhw = ((hmask3 * diff).mean('time'))

    dur_mhw = dur_mhw.rename('duration').assign_attrs(units='days per year')
    mag_mhw = mag_mhw.rename('magnitude').assign_attrs(units='degrees Celsius')

    result = xr.merge([dur_mhw, mag_mhw])

    return result

In [6]:
def save_results(result_ds, period, oax_var):
    file_path = f"/g/data/ia39/ncra/ocean/{oax_var}_{period}_oax1.nc"
    result_ds.to_netcdf(file_path)
    print(f"Saved results to {file_path}")

In [7]:
def process_all_periods1(GWL_periods, sst, ssth, oax_var, threshold90, threshold90h):
    for period, (start, end) in GWL_periods.items():
        if period == 'current':
            result_ds = calculate_heatwave_metrics(ssth, threshold90h, period, start, end).compute()
        else:
            result_ds = calculate_heatwave_metrics(sst, threshold90, period, start, end).compute()
        
        save_results(result_ds, period, oax_var)

In [8]:
# parameters

oax_var='PH'
oax_var='OAR'
oax_var='hco3r'

In [9]:
# SST files
coords_to_drop = ['st_edges_ocean', 'nv', 'st_ocean']
vars_to_drop = ['Time_bounds', 'average_DT', 'average_T1', 'average_T2', 'st_ocean']
# OA files 
coords_to_drop =['ST_OCEAN']
vars_to_drop =['XT_OCEAN_bnds','XT_OCEAN_bnds']

# preprocesser to drop unwanted variables
def drop_stuff(ds, coords_to_drop,vars_to_drop):
    """
    Preprocessor function to drop specified coordinates and variables from a dataset loaded via xr.open_mfdataset

    Parameters:
        ds (xarray.Dataset): The dataset from which coordinates & variables are to be dropped.
        coords_to_drop (list of str): List of coordinate names to drop.
        vars_to_drop(list of str): List of variable names to drop

    Returns:
        xarray.Dataset: Dataset with specified coordinates and variables dropped.
    """
    # Drop coordinates if they are in the dataset
    ds = ds.drop_vars(coords_to_drop, errors='ignore')
    ds = ds.drop_vars(vars_to_drop, errors='ignore')
    return ds

In [10]:
%%time
warnings.filterwarnings('ignore')


# Directory paths for OAE
dir1_new = '/scratch/xv83/rxm599/historical/'
dir2_new = '/scratch/xv83/rxm599/future/'
# modified to get append historical data to start of future output
pattern0 = sorted(glob(dir1_new + 'afiles*.nc'))
pattern1 = sorted(glob(dir1_new + 'afiles201[4-9]*.nc'))
pattern2 = sorted(glob(dir2_new + 'afiles*.nc'))

GWL_periods = {
   'current': ('1995-01-01', '2014-12-31'),
   'GW1p2': ('2001-01-01', '2020-12-31'),
   'GW1p5': ('2015-01-01', '2034-12-31'),
   'GW2p0': ('2030-01-01', '2049-12-31'),
   'GW3p0': ('2053-01-01', '2072-12-31'),
   'GW4p0': ('2074-01-01', '2093-12-31')
}
GWL_periods = {
   'GW2p0': ('2030-01-01', '2049-12-31'),
   'GW3p0': ('2053-01-01', '2072-12-31'),
}


# load threshold data
threshold90h = xr.open_dataset(f'/g/data/ia39/ncra/ocean/mhw/{oax_var}_percentile_sdaily_current.nc')['smooth90']
threshold90h = threshold90h.chunk({'dayofyear': 31, 'xt_ocean': 500, 'yt_ocean': 600})
threshold90 = xr.open_dataset(f'/g/data/ia39/ncra/ocean/mhw/{oax_var}_percentile_sdaily_GW1p5.nc')['smooth90']
threshold90 = threshold90.chunk({'dayofyear': 31, 'xt_ocean': 500, 'yt_ocean': 600})
        
sst,ssth = load_data1(oax_var, coords_to_drop, vars_to_drop, pattern0, pattern1, pattern2)
sst=sst.chunk({'time':31,'xt_ocean': 500, 'yt_ocean': 600})
ssth=ssth.chunk({'time':31,'xt_ocean': 500, 'yt_ocean': 600})


CPU times: user 9.75 s, sys: 1.28 s, total: 11 s
Wall time: 23.7 s


In [11]:
print(threshold90h)
print(sst)
print(ssth)

<xarray.DataArray 'smooth90' (dayofyear: 366, yt_ocean: 1500, xt_ocean: 3600)> Size: 16GB
dask.array<xarray-<this-array>, shape=(366, 1500, 3600), dtype=float64, chunksize=(31, 600, 500), chunktype=numpy.ndarray>
Coordinates:
  * dayofyear  (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * yt_ocean   (yt_ocean) float64 12kB -74.95 -74.85 -74.75 ... 74.85 74.95
  * xt_ocean   (xt_ocean) float64 29kB 0.05 0.15 0.25 0.35 ... 359.8 359.9 360.0
    quantile   float64 8B ...
<xarray.DataArray 'hco3r' (time: 37332, yt_ocean: 1500, xt_ocean: 3600)> Size: 2TB
dask.array<rechunk-merge, shape=(37332, 1500, 3600), dtype=float64, chunksize=(31, 600, 500), chunktype=numpy.ndarray>
Coordinates:
  * time      (time) datetime64[ns] 299kB 1999-12-30T12:00:00 ... 2101-12-09T...
  * yt_ocean  (yt_ocean) float64 12kB -74.95 -74.85 -74.75 ... 74.75 74.85 74.95
  * xt_ocean  (xt_ocean) float64 29kB 0.05 0.15 0.25 0.35 ... 359.8 359.9 360.0
Attributes:
    long_name:                         

In [12]:
%%time
process_all_periods1(GWL_periods, sst*(-1), ssth*(-1), oax_var,threshold90, threshold90h)


2026-02-09 17:55:11,493 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 14.44 GiB -- Worker memory limit: 17.88 GiB
2026-02-09 17:55:17,520 - distributed.worker.memory - WARNING - Worker is at 73% memory usage. Resuming worker. Process memory: 13.15 GiB -- Worker memory limit: 17.88 GiB
2026-02-09 17:55:37,097 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 14.39 GiB -- Worker memory limit: 17.88 GiB
2026-02-09 17:55:37,103 - distributed.worker.memory - WARNING - Worker is at 75% memory usage. Resuming worker. Process memory: 13.46 GiB -- Worker memory limit: 17.88 GiB
2026-02-09 18:01:51,743 - distributed.worker.memory - WARNING - gc.collect() took 1.127s. This is usually a sign that some tasks handle too many Python objects at the same time. Rechunking the work into smaller tasks might help.


Saved results to /g/data/ia39/ncra/ocean/hco3r_GW2p0_oax1.nc


2026-02-09 18:11:54,319 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 14.37 GiB -- Worker memory limit: 17.88 GiB
2026-02-09 18:11:58,116 - distributed.worker.memory - WARNING - Worker is at 74% memory usage. Resuming worker. Process memory: 13.30 GiB -- Worker memory limit: 17.88 GiB
2026-02-09 18:14:05,021 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 14.37 GiB -- Worker memory limit: 17.88 GiB
2026-02-09 18:14:06,803 - distributed.worker.memory - WARNING - Worker is at 73% memory usage. Resuming worker. Process memory: 13.21 GiB -- Worker memory limit: 17.88 GiB
2026-02-09 18:16:59,651 - distributed.worker.memory - WARNING - gc.collect() took 1.110s. This is usually a sign that some tasks handle too many Python objects at the same time. Rechunking the work into smaller tasks might help.


Saved results to /g/data/ia39/ncra/ocean/hco3r_GW3p0_oax1.nc
CPU times: user 15min 26s, sys: 1min 26s, total: 16min 53s
Wall time: 32min 35s


In [13]:
sys.exit(1)

SystemExit: 1

## Play

In [ ]:
GWL_periods = {
   'GW1p5': ('2015-01-01', '2034-12-31'),
}
# select a point to test the MHW code
xlat=-60; xlon= 155
sst1 = sst.sel(yt_ocean=xlat, xt_ocean=xlon, method='nearest')
ssth1 = ssth.sel(yt_ocean=xlat, xt_ocean=xlon, method='nearest')

In [ ]:
%%time
for period, (start, end) in GWL_periods.items():
    threshold90a = threshold90.sel(yt_ocean=xlat, xt_ocean=xlon, method='nearest')
    sst_period = sst1.sel(time=slice(start, end))
    result_ds = calculate_heatwave_metrics(sst1*(-1), threshold90a, period, start, end).compute()
    expanded_threshold = threshold90a.sel(dayofyear=sst_period['time'].dt.dayofyear)
    expanded_threshold = expanded_threshold.drop('dayofyear')

In [ ]:
(sst_period*(-1)).plot()
expanded_threshold.plot()
result_ds.values

In [ ]:
sst_period.plot()


In [ ]:
client.close()